# Transaction types

One example row per type of securities lending transaction, from the cleaned table on the latest reference period.

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

Which types are there and how frequent are they?

In [ ]:
query = f"""

SELECT collateral_type, COUNT(*) AS n
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
GROUP BY 1
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

One example per type. When, who, what was lent, against what, at what price. The collateral column lists the ISINs of securities collateral or the currency and amount of cash collateral.

In [ ]:
query = f"""

SELECT x.reference_period, x.start_date, x.lender_id, x.borrower_id, x.agent_lender_id,
       x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
       x.collateral_type,
       GROUP_CONCAT(CASE WHEN c.collateral_kind = 'cash'
                         THEN CONCAT(c.cash_currency, ' ', CAST(ROUND(c.cash_amount) AS STRING))
                         ELSE c.collateral_isin END, ', ') AS collateral,
       x.lending_fee, x.rebate_rate
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY collateral_type ORDER BY uti) AS rn
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
) x
LEFT JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
  ON c.tec_ruti = x.tec_ruti AND c.reference_period = x.reference_period
WHERE x.rn = 1
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14
ORDER BY x.collateral_type

"""
df = pd.read_sql_query(query, cnxn)
df